This notebook serves as a proof of concept for searching spikes in place names to identify events.
The last cell [`places_1577.ipynb`](../notebooks/places_1577.ipynb) uncovers Scotland as occuring significantly in Edward I.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
import re
import unicodedata
import textwrap

# Helpers: the text and every searched name go through the same spelling normalization

def norm(word):
    """Spelling key for one word: Edwarde -> Edward, VVestmynster -> Westminster."""
    word = word.replace("ã", "an").replace("õ", "on").replace("ẽ", "en").replace("ũ", "un")
    word = unicodedata.normalize("NFKD", word).encode("ascii", "ignore").decode()
    key = word.lower().replace("vv", "w").replace("j", "i").replace("v", "u").replace("y", "i")
    key = re.sub(r"au(?=n)", "a", key)  # Fraunce -> France
    key = re.sub(r"(.)\1", r"\1", key)  # collapse doubled letters (Scotte -> Scote)
    key = re.sub(r"e$", "", key)  # drop a final "e" (Scotlande -> Scotland)
    # Restore the capital so one-word names can still be told apart from common words
    return key.capitalize() if word[:1].isupper() else key

def norm_text(text):
    """Normalize each word of a string (digits and punctuation kept) and collapse whitespace."""
    return " ".join(re.sub(r"[^\W\d_]+", lambda match: norm(match.group()), text).split())

def name_pattern(spellings):
    """Regex for a "|"-separated list of spellings, matched as whole words in normalized text.
    One-word names need a capital (skips common words); multi-word names ignore case."""
    keys = sorted({norm_text(s) for s in spellings.split("|") if s.strip()}, key=len, reverse=True)
    return re.compile(r"(?<!\w)(?:" + "|".join(
        f"(?i:{re.escape(k)})" if " " in k else k[0].upper() + f"(?i:{re.escape(k[1:])})" for k in keys
    ) + r")(?!\w)")

def word_index(text, char_pos):
    """Word position (0-based) of a character offset in `text`."""
    return len(text[:char_pos].split())

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "1577_files").is_dir():
    repo_root = repo_root.parent

# Locate the (Edward) XML file
edward_path = repo_root / "1577_files" / "regnal_files" / "09_EdwardFirst.xml"

TEI = "{http://www.tei-c.org/ns/1.0}"

root = ET.parse(edward_path).getroot()

# Find Edward I's reign section
edward_entry = root.find(f".//{TEI}div[@type='reign'][@n='Edward I']")

if edward_entry is None:
    raise ValueError("Could not find Edward I's reign section.")

# Extract all text in the reign (including margin notes) and normalize it
edward_xml_words = norm_text(" ".join(edward_entry.itertext()))

print(textwrap.fill(edward_xml_words[:1000], width=150))

In [ ]:
# Split into list of words (the x-axis unit for every chart below)
edward_words = edward_xml_words.split()

# Find occurrences of "Scotland" and "Scotlande" (the text is normalized, so both read "Scotland")
scotland_pattern = re.compile(r"(?<!\w)Scotland(?:e)?(?!\w)", re.IGNORECASE)

# Word position of each mention
word_positions = [
    word_index(edward_xml_words, match.start())
    for match in scotland_pattern.finditer(edward_xml_words)
]

print(f"Found {len(word_positions)} mentions of Scotland/Scotlande.")

# Count mentions in consecutive 500-word sections
window_size = 500
number_of_windows = (len(edward_words) + window_size - 1) // window_size

scotland_counts = (
    pd.Series(word_positions)
    .floordiv(window_size)
    .value_counts()
    .sort_index()
    .reindex(range(number_of_windows), fill_value=0)
)

# Extract marginal years (1272-1307, Edward I's reign) and map each to its word position in the text
year_pattern = re.compile(r"(?<!\d)(?:12(?:7[2-9]|[89]\d)|130[0-7])(?!\d)")
year_markers = []  # (year, word position) pairs
search_from = 0  # notes appear in order, so each search starts after the previous note

for note in edward_entry.findall(f".//{TEI}note[@place='margin']"):
    # Normalize the same way as the full text so the note can be found inside it
    note_text = norm_text(" ".join(note.itertext()))
    matches = list(year_pattern.finditer(note_text))
    if not matches:
        continue

    note_offset = edward_xml_words.find(note_text, search_from)
    if note_offset < 0:
        raise ValueError(f"Could not locate marginal note in extracted text: {note_text!r}")

    for match in matches:
        year_markers.append((int(match.group()), word_index(edward_xml_words, note_offset + match.start())))

    search_from = note_offset + len(note_text)

display(scotland_counts.sort_values(ascending=False).head(10))

# Plot mentions across the text
plt.figure(figsize=(16, 5))

plt.bar(
    scotland_counts.index * window_size,
    scotland_counts.values,
    width=window_size * 0.9,
    align="edge",
)

ax = plt.gca()
max_count = scotland_counts.max()
for i, (year, word_position) in enumerate(year_markers):
    ax.axvline(word_position, color="darkred", alpha=0.25, linewidth=0.8)
    ax.text(
        word_position,
        max_count + 1 + (i % 4) * 2.5,
        str(year),
        rotation=90,
        ha="center",
        va="bottom",
        color="darkred",
        fontsize=8,
    )
ax.set_ylim(0, max_count + 13)

plt.xlabel("Word in Edward I's text")
plt.ylabel("Mentions per 500-word section")
plt.title("Scotland/Scotlande Mentions Across Edward I's Reign")
plt.xticks(ticks=range(0, len(edward_words) + 1, 2000), rotation=45)
plt.tight_layout()
plt.show()

We see a spike between words 11000-17000

In [ ]:
# Load the people CSV (columns: name, spelling_variants, example_entry)
people_df = pd.read_csv(repo_root / "name_extraction" / "entity_names" / "holinshed_people.csv")

# Merge rows whose normalized names match (e.g., Malcolm / Malcolme); the first name becomes the label.
# spelling_variants is skipped: for people it holds short forms like "Iohn" that match too widely
people_df = people_df.groupby(people_df["name"].map(norm_text).str.lower(), sort=False).agg(
    name=("name", "first"), spellings=("name", "|".join)
)
# One compiled regex per person, reused for counting and for plotting positions
patterns = {row["name"]: name_pattern(row["spellings"]) for _, row in people_df.iterrows()}

# Extract the text slice (words 11000-17000) to search for people in
eleven_seventeen = " ".join(edward_words[11000:17000])

# Count occurrences of each person's name
people_df["count"] = people_df["name"].map(
    lambda name: len(patterns[name].findall(eleven_seventeen))
)

# Show people who appear, ordered by count
people_in_slice = (
    people_df[people_df["count"] > 0]
    [["name", "count"]]
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(people_in_slice.head(10))



In [ ]:
# Reproduce previous cell with top names to see where they are mentioned
slice_start = 11000
slice_end = 17000
top_people = people_in_slice.head(8)["name"].tolist()

# Word position of every mention, offset by slice_start so it lines up with the full-text axis
person_positions = {
    name: [
        slice_start + word_index(eleven_seventeen, m.start())
        for m in patterns[name].finditer(eleven_seventeen)
    ]
    for name in top_people
}

fig, (ax_bar, ax_people) = plt.subplots(
    2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)

# Top panel: identical to the Cell 3 bar chart (same data, bins, and styling)
ax_bar.bar(
    scotland_counts.index * window_size,
    scotland_counts.values,
    width=window_size * 0.9,
    align="edge",
)
ax_bar.set_ylabel("Mentions per 500-word section")
ax_bar.set_title("Scotland/Scotlande Mentions Across Edward I's Reign")

# Bottom panel: one row (a "swimlane") per person, on the same word axis as above
colors = plt.cm.tab10.colors
for i, (name, positions) in enumerate(person_positions.items()):
    ax_people.eventplot(
        positions,
        lineoffsets=i,
        linelengths=0.8,
        linewidths=2,
        colors=colors[i % len(colors)],
    )

ax_people.set_yticks(range(len(person_positions)))
ax_people.set_yticklabels(list(person_positions.keys()))
ax_people.set_ylim(-0.5, len(person_positions) - 0.5)
ax_people.invert_yaxis()
ax_people.set_xlabel("Word in Edward I's text")
ax_people.set_xticks(range(slice_start, slice_end + 1, 500))
ax_people.tick_params(axis="x", rotation=45)

# Zoom both panels (they share the x-axis) into the slice the names were searched in
ax_people.set_xlim(slice_start, slice_end)

fig.tight_layout()
plt.show()


In [ ]:
# Quickly examine where a few people appear
section = " ".join(edward_words[11750:12000])
print(textwrap.fill(section, width=150))